This notebook was run last on the following commit

In [ ]:
!git log -1

In [ ]:
%matplotlib inline

In [ ]:
import scanpy as sc

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as gg
import numpy as np
import sys

sys.path.append("/workspace")


from src.evaluation.kernel_evaluation import (
    compute_distance_matrix,
    process_and_align,
    wide_to_long,
)

In [ ]:
# metabolic_kernel_path = "/workspace/results/ecoli_rich_medium/fba_moma/default/kernel.pkl"
# metabolic_dist_path = "/workspace/results/ecoli_rich_medium/gene_graph/beta_auto/distances.pkl"
metabolic_dist_path = "/workspace/results/ecoli_rich_medium/gene_graph/beta_1/distances.pkl"
reference_dist_path = "/workspace/results/ecoli_rich_medium/targets/mmd_distances.pkl"

metabolic_dist = pd.read_pickle(metabolic_dist_path)
reference_dist = pd.read_pickle(reference_dist_path)

metabolic_dist_, reference_dist_ = process_and_align(metabolic_dist, reference_dist)

metabolic_dist_long = wide_to_long(
    metabolic_dist_,
    "distance_pred",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)

reference_dist_long = wide_to_long(
    reference_dist_,
    "distance_target",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
joint_long = pd.merge(
    metabolic_dist_long, reference_dist_long, on=["gene1", "gene2"], how="inner"
).assign(gene_pair=lambda x: x["gene1"] + "-" + x["gene2"])

In [ ]:
plt.scatter(joint_long["distance_pred"], joint_long["distance_target"], alpha=0.5)

In [ ]:
import plotly.express as px


fig = px.scatter(
    joint_long.query("distance_pred < 0.05"),
    x="distance_pred",
    y="distance_target",
    hover_name="gene_pair",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
joint_long.query("distance_pred ==0.00")

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

adata.obs["umap_x"] = adata.obsm["X_umap"][:, 0]
adata.obs["umap_y"] = adata.obsm["X_umap"][:, 1]

In [ ]:
gene1 = "fbaA"
gene2 = "pfkA"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)
display(fig)

In [ ]:
filter_d = joint_long.query("distance_pred < 0.75")
plt.hist(filter_d["distance_target"], bins=25)

In [ ]:
filter_d.hist(column="distance_target", bins=25)

In [ ]:
print("; ".join(filter_d.sample(frac=1, replace=False)["gene_pair"]))

In [ ]:
filter_d.sort_values("distance_target", ascending=False).head(50)

In [ ]:
gene1 = "cyaA"
gene2 = "dosP"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)
display(fig)